# All logic to embbeding movie

#### Variables a conciderar
- adult (2)
- original_lenguaje (4)
- title (3)
- keywords (5)
- popularity (3)
- date (year) (3)
- director (5)
- actors (5)
- vote_average (3)
- vote_count (3)
- Genre (7)
-------------------------
43

## Connect to DB

In [1]:
import mysql.connector
import pandas as pd
#-----------------------------
import random
import mmh3
import numpy as np
#-----------------------------
import gensim.downloader as api
import ast
#-----------------------------
from annoy import AnnoyIndex
import faiss

In [2]:
config = {
    'host': '127.0.0.1',
    'port':'3307',
    'user': 'REG',
    'password': 'Aa123456',
    'database': 'TV_MOVIES_DB'
}

In [9]:
# conexion = mysql.connector.connect(**config)
# cursor = conexion.cursor()
# consulta = "SELECT * FROM Movies WHERE YEAR(release_date) = 2021;"
# cursor.execute(consulta)

# results = cursor.fetchall()
# column_names = [desc[0] for desc in cursor.description]

# movie_df = pd.DataFrame(results, columns=column_names)

In [11]:
conexion = mysql.connector.connect(**config)
cursor = conexion.cursor()
consulta = "SELECT * FROM Movies"
cursor.execute(consulta)

results = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

movie_df = pd.DataFrame(results, columns=column_names)

In [12]:
movie_df.head(2)

,id,id_tmdb,adult,backdrop_path,original_lenguaje,overview,keywords,popularity,poster_path,release_date,title,director,actors,vote_average,vote_count,image_path,created_at
0,1,565426,0,/txaC2mhUhW0H0r31N6KzRTaz1rL.jpg,en,Lara Jean and Peter have just taken their roma...,"['lara jean', 'romance pretend', 'real recipie...",3.951,/maib5VlmEqp5xlN8lptnBSftp2o.jpg,2020-02-03,To All the Boys: P.S. I Still Love You,Michael Fimognari,"['Lana Condor', 'Noah Centineo', 'Jordan Fishe...",6.822,2669,Data/img_recommendMe/565426_to_all_the_boys_ps...,2026-02-10 17:02:24


In [13]:
batman_movie = movie_df.iloc[110]
batman_movie

IndexError: single positional indexer is out-of-bounds

## Get all vectors individualy

### Get adult vector

In [14]:
def multiply_num_vector(num, dimensions=2):
    vector = []

    for i in range(dimensions):
        vector.append(float(num))

    return vector

In [10]:
multiply_num_vector(batman_movie['adult'],2)

[0.0, 0.0]

### Get original lenguaje vector

In [15]:
def hashing_trick(text, dimensions=64, seed_offset=0):

    if text == "" or text == None or text == []:
        return np.zeros(dimensions, dtype=np.float32)

    if not text or not isinstance(text, str):
        text = str(text)
    
    hash_num = mmh3.hash(text, seed=42)
    
    vector = np.zeros(dimensions, dtype=np.float32)
    
    prime1 = 2654435761  
    prime2 = 5915587277
    
    for i in range(dimensions):

        seed = (hash_num ^ (i * prime1) ^ (seed_offset * prime2)) & 0xFFFFFFFF
        
        rng = np.random.RandomState(seed)

        value = rng.uniform(low=-10.0, high=10.0)
        vector[i] = value
    
    return vector

In [12]:
hashing_trick(batman_movie['original_lenguaje'],4)

array([ 3.4582613, -8.665419 , -4.83918  ,  9.013555 ], dtype=float32)

### Get title vector

In [16]:
def hashing_trick_list(list_text, dimensions=64, seed_offset=0):

    if list_text == "" or list_text == None or list_text == []:
        return np.zeros(dimensions, dtype=np.float32)

    text_vectors = []
    for text in list_text:
        if not text or not isinstance(text, str):
            text = str(text)
        
        hash_num = mmh3.hash(text, seed=42)
        
        vector = np.zeros(dimensions, dtype=np.float32)
        
        prime1 = 2654435761  
        prime2 = 5915587277
        
        for i in range(dimensions):
    
            seed = (hash_num ^ (i * prime1) ^ (seed_offset * prime2)) & 0xFFFFFFFF
            
            rng = np.random.RandomState(seed)
    
            value = rng.uniform(low=-10.0, high=10.0)
            vector[i] = value
            text_vectors.append(vector)
    
    return np.mean(text_vectors, axis=0)

In [14]:
list_title = list(batman_movie['title'])
hashing_trick_list(list_title,3)

array([-0.09674486,  0.4463041 , -3.394736  ], dtype=float32)

### Get keywords vector

In [22]:
model = api.load("glove-wiki-gigaword-50")

In [17]:
def top_words_5_vector(list_text, reducir_a_5D=True):

    if list_text == "" or list_text == None or list_text == []:
        return np.zeros(5, dtype=np.float32)

    top_5_vectors = []
    for text in list_text:
    
        words = text.lower().split()
        words_vectors = []
    
        for word in words:
            try:
                vector_50d = model[word]
            except KeyError:
                vector_50d = np.zeros(50)
            
            if not reducir_a_5D:
                return vector_50d
            
            vector_5d = np.zeros(5)
            bloque = 50 // 5 
    
            for i in range(5):
                inicio = i * bloque
                fin = inicio + bloque
                vector_5d[i] = np.mean(vector_50d[inicio:fin])
                
            words_vectors.append(vector_5d)
        top_5_vectors.append(np.mean(words_vectors, axis=0))
        
    return np.round(np.mean(top_5_vectors, axis=0), 7)

In [17]:
keywords_list = ast.literal_eval(batman_movie['keywords'])
top_words_5_vector(keywords_list)

array([ 0.0737301,  0.0821995, -0.0836377,  0.0037927, -0.0998564])

### Get popularity vector

In [18]:
print(multiply_num_vector(batman_movie['popularity'], 3))

[1.482, 1.482, 1.482]


### Get date vector

In [18]:
def date_vector(release_date, num=3):
    year = str(release_date).split('-')[0]
    year = int(year) - 2000
    return multiply_num_vector(year, num)

In [20]:
date_vector(batman_movie['release_date'])

[20.0, 20.0, 20.0]

### Get director vector

In [21]:
hashing_trick(batman_movie['director'] ,5)

array([7.68838  , 2.8533568, 5.6163015, 4.477386 , 1.2732053],
      dtype=float32)

### Get actors vector

In [22]:
actors_list = ast.literal_eval(batman_movie['actors'])
hashing_trick_list(actors_list,5)

array([-0.95966446,  2.394908  , -0.9665145 , -2.945136  , -1.0375808 ],
      dtype=float32)

### Get vote_average vector

In [23]:
multiply_num_vector(batman_movie['vote_average'], 3)

[6.6, 6.6, 6.6]

### Get vote_count

In [24]:
vote_count = float(batman_movie['vote_count']) / 1000
multiply_num_vector(vote_count, 3)

[0.064, 0.064, 0.064]

### Get gender vector

In [19]:
def get_gender_movie(movie_id, dimension=7):

    conexion = mysql.connector.connect(**config)
    cursor = conexion.cursor()
    consulta = "SELECT genre_id FROM Movie_Genres Where movie_id = " + str(movie_id)
    cursor.execute(consulta)
    
    results = cursor.fetchall()
    gender = [x[0] for x in results]
    if gender == "" or gender == None:
        return np.zeros(dimension, dtype=np.float32)
    return hashing_trick_list(gender,dimension)

In [26]:
get_gender_movie(batman_movie["id"])

array([ 2.449926  ,  3.220671  , -6.8632817 ,  4.2147765 ,  1.019425  ,
       -0.12754276,  2.0236647 ], dtype=float32)

### Generate new Dataframe Vectorized

In [20]:
def safe_literal_eval(x):
    if pd.isna(x):
        return []
    
    if isinstance(x, list):
        return x  # Ya es lista
    
    if isinstance(x, str):
        x = x.strip()
        if x and x.startswith('[') and x.endswith(']'):
            try:
                return ast.literal_eval(x)
            except:
                pass
        return [x] if x else []
    
    return []

In [23]:
df_vectorized = pd.DataFrame({
    "id": movie_df['id'],
    'adult': movie_df['adult'].apply(lambda x: multiply_num_vector(x, 2)),
    'original_lenguaje': movie_df['original_lenguaje'].apply(lambda x: hashing_trick(x, 4)),
    'title': movie_df['title'].apply(lambda x: hashing_trick_list(x, 3)),
    'keywords': movie_df['keywords'].apply(lambda x: top_words_5_vector(ast.literal_eval(x))),
    'popularity': movie_df['popularity'].apply(lambda x: multiply_num_vector(x, 3)),
    'release_date': movie_df['release_date'].apply(lambda x: date_vector(x, 3)),
    'director': movie_df['director'].apply(lambda x: hashing_trick(x, 5)),
    'actors': movie_df['actors'].apply(lambda x: hashing_trick_list(safe_literal_eval(x), 5)),
    'vote_average': movie_df['vote_average'].apply(lambda x: multiply_num_vector(x, 3)),
    'vote_count': movie_df['vote_count'].apply(lambda x: multiply_num_vector(float(x) / 1000, 3)),
    'gender': movie_df['id'].apply(lambda x: get_gender_movie(x,7))
})

vector_columns = ['adult','original_lenguaje','title','keywords','popularity',
                 'release_date', 'director', 'actors', 'vote_average',
                 'vote_count', 'gender']
df_vectorized['movie_vector'] = df_vectorized[vector_columns].apply(
    lambda row: np.concatenate([row[col] for col in vector_columns]),
    axis=1
)

In [24]:
df_vectorized.head()

,id,adult,original_lenguaje,title,keywords,popularity,release_date,director,actors,vote_average,vote_count,gender,movie_vector
0,1,"[0.0, 0.0]","[3.4582613, -8.665419, -4.83918, 9.013555]","[-2.15566, -2.3455803, 1.3848987]","[0.0624972, 0.0605411, -0.0829374, 0.1231957, ...","[3.951, 3.951, 3.951]","[20.0, 20.0, 20.0]","[-0.5351226, -7.3357987, 8.158843, -9.692719, ...","[-3.5462797, 3.3527627, -0.89187896, 1.8922313...","[6.822, 6.822, 6.822]","[2.669, 2.669, 2.669]","[4.295691, -3.1224782, -0.46065953, -0.0377524...","[0.0, 0.0, 3.458261251449585, -8.6654186248779..."


In [25]:
len(df_vectorized['movie_vector'].iloc[0])

43

In [26]:
df_vectorized['movie_vector'].iloc[0]

array([ 0.        ,  0.        ,  3.45826125, -8.66541862, -4.83917999,
        9.01355457, -2.15565991, -2.34558034,  1.38489866,  0.0624972 ,
        0.0605411 , -0.0829374 ,  0.1231957 ,  0.0514495 ,  3.951     ,
        3.951     ,  3.951     , 20.        , 20.        , 20.        ,
       -0.53512257, -7.33579874,  8.15884304, -9.69271946, -9.40436363,
       -3.54627967,  3.3527627 , -0.89187896,  1.89223135,  0.9273572 ,
        6.822     ,  6.822     ,  6.822     ,  2.669     ,  2.669     ,
        2.669     ,  4.29569101, -3.12247825, -0.46065953, -0.03775242,
       -4.18693304, -8.21774006, -3.72150183])

In [27]:
df_vectorized.shape

(1, 13)

## Test CSV

In [30]:
movie_df = pd.read_csv('../movies_db.csv')

In [32]:
movie_df.head(2)

,id,id_tmdb,adult,backdrop_path,original_lenguaje,overview,keywords,popularity,poster_path,release_date,title,director,actors,vote_average,vote_count,image_path,created_at
0,1,522627,0,/tintsaQ0WLzZsTMkTiqtMB3rfc8.jpg,en,American expat Mickey Pearson has built a high...,"['profitable marijuana', 'expat mickey', 'lond...",8.102,/jtrhTYB7xSrJxR1vusu99nvnZ1g.jpg,2020-01-01,The Gentlemen,Guy Ritchie,"['Matthew McConaughey', 'Charlie Hunnam', 'Mic...",7.670,6379,Data/img_recommendMe/522627_the_gentlemen.jpg,2026-01-29 15:07:32
1,2,593402,0,/8TiYxxck4kfKwXAAdi6aZLeQd5L.jpg,it,Checco is a young Apulian entrepreneur dreamer...,"['checco', 'apulian entrepreneur', 'sushi rest...",7.025,/CqUxog8F6aaK97RYh8YXhv3NDL.jpg,2020-01-01,Tolo Tolo,Checco Zalone,"['Checco Zalone', 'Manda Touré', 'Nassor Said ...",6.111,1296,Data/img_recommendMe/593402_tolo_tolo.jpg,2026-01-29 15:07:35


In [34]:
for index, row in movie_df.iterrows():
    print(row['id'], row['id_tmdb'], row['original_lenguaje'])

1 522627 en
2 593402 it
3 430155 ru
4 448119 en
5 630220 it
6 609031 uk
7 666180 en
8 646453 en
9 656844 en
10 639289 en
11 443791 en
12 575426 it
13 592834 en
14 590869 it
15 632309 en
16 526019 en
17 584850 hi
18 607313 te
19 527534 en
20 631430 en
21 550738 fr
22 628241 te
23 662844 en
24 38700 en
25 586461 en
26 575718 ko
27 613345 fr
28 613319 fr
29 474764 en
30 633172 de
31 609242 es
32 620924 en
33 639798 es
34 573730 ja
35 665090 tr
36 651070 en
37 642700 pl
38 566397 ko
39 604822 zh
40 487631 fr
41 604362 ko
42 653574 en
43 596247 es
44 652483 pt
45 503917 cn
46 492611 en
47 442065 en
48 603770 zh
49 653643 en
50 522241 en
51 619918 no
52 548473 en
53 653723 en
54 666959 sv
55 656690 en
56 653601 en
57 552532 en
58 575774 en
59 532870 en
60 653744 en
61 567970 en
62 589970 fr
63 582922 fr
64 451184 en
65 621749 en
66 606954 it
67 648990 es
68 542224 en
69 664593 tr
70 627463 en
71 466622 en
72 528761 en
73 653567 en
74 631132 ja
75 565426 en
76 495764 en
77 571625 ko
78 649755

#### Variables a conciderar
- adult (2)
- original_lenguaje (4)
- title (3)
- keywords (5)
- popularity (3)
- date (year) (3)
- director (5)
- actors (5)
- vote_average (3)
- vote_count (3)
- Gender (7)
-------------------------
43

## Validate distance between movies

In [35]:
def crear_indice_annoy(vectores, n_arboles=10):
    """Crea índice Annoy"""
    dimension = vectores.shape[1]
    idx = AnnoyIndex(dimension, 'angular')  # 'angular' = similitud coseno
    
    for i, vec in enumerate(vectores):
        idx.add_item(i, vec)
    
    idx.build(n_arboles)
    return idx

def buscar_similares_annoy(indice, vector, k=5):
    """Busca k vecinos más cercanos"""
    return indice.get_nns_by_vector(vector, k, include_distances=True)

In [36]:
indice = crear_indice_annoy(np.vstack(df_vectorized['movie_vector'].values))

In [37]:
batman_vector_test = df_vectorized.iloc[110]['movie_vector']
buscar_similares_annoy(indice, batman_vector_test,5)

([110, 3718, 3295, 1309, 1949],
 [0.000295598671073094,
  0.2637985050678253,
  0.3083724081516266,
  0.31121954321861267,
  0.31370118260383606])

In [40]:
movie_df.iloc[143]

id                                                                 144
id_tmdb                                                         502425
adult                                                                0
backdrop_path                         /vwbmp3vvX4U0VjinaQktYOBt5kW.jpg
original_lenguaje                                                   en
overview             South Africa, 1978. Tim Jenkin and Stephen Lee...
keywords             ['imprisoned apartheid', 'lee white', 'tim jen...
popularity                                                       1.980
poster_path                           /8GGS0jkFFCnmdStvZED6NL6V7gd.jpg
release_date                                                2020-03-06
title                                             Escape from Pretoria
director                                                 Francis Annan
actors               ['Daniel Radcliffe', 'Daniel Webber', 'Ian Har...
vote_average                                                     7.180
vote_c

## Make Test

In [41]:
indice = crear_indice_annoy(np.vstack(df_vectorized['movie_vector'].values))

In [42]:
test_movie = movie_df[movie_df['title'].str.contains('Sonic')]
test_movie

,id,id_tmdb,adult,backdrop_path,original_lenguaje,overview,keywords,popularity,poster_path,release_date,title,director,actors,vote_average,vote_count,image_path,created_at
85,86,454626,0,/stmYfCUGd8Iy6kAMBr6AmWqx8Bq.jpg,en,"Powered with incredible speed, Sonic The Hedge...","['sonic hedgehog', 'stop robotnik', 'vs super'...",10.221,/aQvJ5WPzZgYVDrxLX4R6cLJCEaQ.jpg,2020-02-12,Sonic the Hedgehog,Jeff Fowler,"['Ben Schwartz', 'James Marsden', 'Tika Sumpte...",7.292,10226,Data/img_recommendMe/454626_sonic_the_hedgehog...,2026-01-29 15:10:02
1647,1648,675353,0,/xuLA0pii2IMJW2puT7EvJtgpg0H.jpg,en,"After settling in Green Hills, Sonic is eager ...","['hills sonic', 'knuckles search', 'robotnik r...",11.295,/6DrHO1jr3qVrViUO6s6kFiAGM7.jpg,2022-03-30,Sonic the Hedgehog 2,Jeff Fowler,"['Ben Schwartz', 'James Marsden', 'Tika Sumpte...",7.442,5747,Data/img_recommendMe/675353_sonic_the_hedgehog...,2026-01-29 15:50:19
3741,3743,939243,0,/noPEm6Vu9Lm6QUdTGLOyRgk4M6s.jpg,en,"Sonic, Knuckles, and Tails reunite against a p...","['sonic knuckles', 'tails reunite', 'adversary...",18.190,/d8Ryb8AunYAuycVKDp5HpdWPKgC.jpg,2024-12-19,Sonic the Hedgehog 3,Jeff Fowler,"['Jim Carrey', 'Ben Schwartz', 'Keanu Reeves',...",7.632,3118,Data/img_recommendMe/939243_sonic_the_hedgehog...,2026-01-29 16:51:07


In [43]:
movie = movie_df.iloc[85]
movie

id                                                                  86
id_tmdb                                                         454626
adult                                                                0
backdrop_path                         /stmYfCUGd8Iy6kAMBr6AmWqx8Bq.jpg
original_lenguaje                                                   en
overview             Powered with incredible speed, Sonic The Hedge...
keywords             ['sonic hedgehog', 'stop robotnik', 'vs super'...
popularity                                                      10.221
poster_path                           /aQvJ5WPzZgYVDrxLX4R6cLJCEaQ.jpg
release_date                                                2020-02-12
title                                               Sonic the Hedgehog
director                                                   Jeff Fowler
actors               ['Ben Schwartz', 'James Marsden', 'Tika Sumpte...
vote_average                                                     7.292
vote_c

In [44]:
test = df_vectorized.iloc[85]['movie_vector']
buscar_similares_annoy(indice, test,5)

([85, 1647, 1433, 2530, 1823],
 [0.00031008184305392206,
  0.231051966547966,
  0.28900814056396484,
  0.32614538073539734,
  0.34970614314079285])

In [48]:
close_movie = movie_df.iloc[1823]
close_movie

id                                                                1824
id_tmdb                                                         616037
adult                                                                0
backdrop_path                         /jsoz1HlxczSuTx0mDl2h0lxy36l.jpg
original_lenguaje                                                   en
overview             After his retirement is interrupted by Gorr th...
keywords             ['thor odinson', 'god butcher', 'valkyrie korg...
popularity                                                      11.506
poster_path                           /pIkRyD18kl4FhoCNQuWxWu5cBLM.jpg
release_date                                                2022-07-06
title                                           Thor: Love and Thunder
director                                                 Taika Waititi
actors               ['Chris Hemsworth', 'Natalie Portman', 'Christ...
vote_average                                                     6.400
vote_c

### Get CSV from Database

In [32]:
conexion = mysql.connector.connect(**config)
cursor = conexion.cursor()
consulta = "SELECT * FROM Movie_Genres"
cursor.execute(consulta)

results = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

movie_Genres_df = pd.DataFrame(results, columns=column_names)

In [33]:
movie_df.to_csv("../movies_db.csv", index=False)
movie_Genres_df.to_csv("../movie_Genres_df.csv", index=False)
df_vectorized.to_csv("../vectorized_movies_df.csv", index=False)

### Another Test

In [47]:
import numpy as np
import faiss

In [1]:
import numpy as np
import faiss

d = 64
xb = np.random.rand(100, d).astype("float32")
xq = np.random.rand(1, d).astype("float32")

index = faiss.IndexFlatL2(d)
index.add(xb)

D, I = index.search(xq, 5)

print(I)
print(D)

numpy: 1.24.4
faiss: 1.7.4
[[90 19 32  9  7]]
[[6.948475 7.21723  7.513218 7.604233 8.011496]]


In [33]:
import faiss

In [34]:
d = 43
index = faiss.IndexFlatL2(d)
index.add(np.vstack(df_vectorized['movie_vector'].values))

In [55]:
movie = movie_df.iloc[85]
movie

id                                                                  86
id_tmdb                                                         454626
adult                                                                0
backdrop_path                         /stmYfCUGd8Iy6kAMBr6AmWqx8Bq.jpg
original_lenguaje                                                   en
overview             Powered with incredible speed, Sonic The Hedge...
keywords             ['sonic hedgehog', 'stop robotnik', 'vs super'...
popularity                                                      10.221
poster_path                           /aQvJ5WPzZgYVDrxLX4R6cLJCEaQ.jpg
release_date                                                2020-02-12
title                                               Sonic the Hedgehog
director                                                   Jeff Fowler
actors               ['Ben Schwartz', 'James Marsden', 'Tika Sumpte...
vote_average                                                     7.292
vote_c

In [65]:
query = df_vectorized.iloc[85]['movie_vector']
query = np.array(query, dtype=np.float32).reshape(1, -1)

D, I = index.search(query, 15)
print(I)
print(D)

[[  85 1647 1433 1328  858 1823 2530  370 1062  289    3 1275 2549 3741
   933]]
[[  0.      138.53696 207.89484 231.19183 316.73822 324.18002 342.31445
  355.5838  380.36615 389.02966 407.78305 412.7419  418.01248 436.53098
  442.67505]]


In [ ]:
3741

In [69]:
movie = movie_df.iloc[2549]
movie

id                                                                2551
id_tmdb                                                         447277
adult                                                                0
backdrop_path                         /lAs4Y5a8Pi86xDwuv8cx2prOVI2.jpg
original_lenguaje                                                   en
overview             The youngest of King Triton’s daughters, and t...
keywords             ['ariel', 'sea witch', 'triton daughters', 'da...
popularity                                                      11.966
poster_path                           /ym1dxyOk4jFcSl4Q2zmRrA5BEEN.jpg
release_date                                                2023-05-18
title                                               The Little Mermaid
director                                                  Rob Marshall
actors               ['Halle Bailey', 'Jonah Hauer-King', 'Melissa ...
vote_average                                                     6.277
vote_c

In [ ]:
[85, 1647, 1433, 2530, 1823]

In [54]:
movie = movie_df.iloc[1823]
movie

id                                                                1824
id_tmdb                                                         616037
adult                                                                0
backdrop_path                         /jsoz1HlxczSuTx0mDl2h0lxy36l.jpg
original_lenguaje                                                   en
overview             After his retirement is interrupted by Gorr th...
keywords             ['thor odinson', 'god butcher', 'valkyrie korg...
popularity                                                      11.506
poster_path                           /pIkRyD18kl4FhoCNQuWxWu5cBLM.jpg
release_date                                                2022-07-06
title                                           Thor: Love and Thunder
director                                                 Taika Waititi
actors               ['Chris Hemsworth', 'Natalie Portman', 'Christ...
vote_average                                                     6.400
vote_c